In [22]:
import pandas as pd
import os
from collections import defaultdict
from tqdm import tqdm

In [20]:
df = pd.read_csv("processed_data/ncaa_basketball_processed_2003_2023.csv")

# Adding individual columns for attempts and makes
df[['FGM', 'FGA']] = df['field_goals_made_field_goals_attempted'].str.split('-', expand=True).astype(float)
df[['3PM', '3PA']] = df['three_point_field_goals_made_three_point_field_goals_attempted'].str.split('-', expand=True).astype(float)
df[['FTM', 'FTA']] = df['free_throws_made_free_throws_attempted'].str.split('-', expand=True).astype(float)

# Renaming columns to make things more concise
rename_map = {
    "game_id" : "Game_ID",
    "season" : "Season",
    "season_type" : "Season_Type",
    "game_date" : "Date",
    "game_date_time" : "Datetime",
    "team_location" : "Team",
    "opponent_team_location" : "Opponent",
    "home_away_combined" : "Site",
    "team_id" : "Team_ID",
    "opponent_id" : "Opp_ID",
    "team_score" : "Team_Score",
    "opponent_team_score" : "Opp_Score",
    "team_winner" : "Team_Win",
    "largest_lead" : "Largest_Lead",
    "field_goal_pct" : "FG%",
    "three_point_field_goal_pct" : "3P%",
    "free_throw_pct" : "FT%",
    "total_rebounds" : "TRB",
    "offensive_rebounds" : "ORB",
    "assists" : "AST",
    "steals" : "STL",
    "blocks" : "BLK",
    "total_turnovers" : "TOV",
    "fouls" : "PF"
}

df.rename(columns=rename_map, inplace=True)


# Columns we want opponent versions of
stat_cols = ["FGM", "FGA", "FG%", "3PM", "3PA", "3P%", "FTM", "FTA", "FT%", 
             "ORB", "TRB", "AST", "STL", "BLK", "TOV", "PF"]

# Perform self-merge on Game_ID
merged = df.merge(
    df[["Game_ID", "Team"] + stat_cols],
    on="Game_ID",
    suffixes=("", "_opp")
)

# Keep only rows where the opponent team is not the same team
merged = merged[merged["Team"] != merged["Team_opp"]]

# Rename opponent columns to start with 'o'
for col in stat_cols:
    merged.rename(columns={f"{col}_opp": f"o{col}"}, inplace=True)

# Drop the now-unneeded 'Team_opp' column
merged.drop(columns=["Team_opp","field_goals_made_field_goals_attempted","three_point_field_goals_made_three_point_field_goals_attempted","free_throws_made_free_throws_attempted","defensive_rebounds","team_rebounds"], inplace=True)

# Make sure the output folder exists
os.makedirs("processed_year_data", exist_ok=True)

# Split merged DataFrame by season and save each as a separate CSV
for season, group in merged.groupby("Season"):
    # Optional: reset index for each CSV
    group = group.reset_index(drop=True)
    
    # Create a filename per season
    filename = f"processed_year_data/cbb_{season}.csv"
    
    # Save to CSV
    group.to_csv(filename, index=False)
    
    print(f"Saved {filename} with {len(group)} rows.")


C:\Users\dschr\AppData\Local\Temp\ipykernel_22732\311535097.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("processed_data/ncaa_basketball_processed_2003_2023.csv")


Saved processed_year_data/cbb_2003.csv with 2 rows.
Saved processed_year_data/cbb_2004.csv with 24 rows.
Saved processed_year_data/cbb_2005.csv with 8836 rows.
Saved processed_year_data/cbb_2006.csv with 9846 rows.
Saved processed_year_data/cbb_2007.csv with 10488 rows.
Saved processed_year_data/cbb_2008.csv with 11094 rows.
Saved processed_year_data/cbb_2009.csv with 11278 rows.
Saved processed_year_data/cbb_2010.csv with 11496 rows.
Saved processed_year_data/cbb_2011.csv with 11294 rows.
Saved processed_year_data/cbb_2012.csv with 11288 rows.
Saved processed_year_data/cbb_2013.csv with 11368 rows.
Saved processed_year_data/cbb_2014.csv with 11640 rows.
Saved processed_year_data/cbb_2015.csv with 11628 rows.
Saved processed_year_data/cbb_2016.csv with 11646 rows.
Saved processed_year_data/cbb_2017.csv with 11610 rows.
Saved processed_year_data/cbb_2018.csv with 11798 rows.
Saved processed_year_data/cbb_2019.csv with 11876 rows.
Saved processed_year_data/cbb_2020.csv with 11512 rows.
S

In [28]:
# Define columns for cumulative stats
columns = ['Name', 'Game', 'Opponent', 'Site', 'Outcome', 'Date', 'ORtg', 'DRtg', '3PAr', 'TS%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'eFG%', 
            'TOV%', 'ORB%', 'FTr', 'oeFG%', 'oTOV%', 'oDRB%', 'oFTr', 'Pace', 'o3PAr']

for i in range(2003, 2024):
    year = str(i)

    if os.path.exists("AdvWeek_CSVs/AdvWeekData" + year + ".csv"):
        continue

    # Read data from CSV
    data = pd.read_csv("processed_year_data/cbb_" + year + ".csv")

    # Initialize/Reset dictionary to store totals data
    team_stats = defaultdict(lambda: {
        'Games': 0, 'Points': 0, 'oPoints': 0, 'FGA': 0, 'FGM': 0, '3PA': 0, '3PM': 0,
        'FTA': 0, 'FTM': 0, 'ORB': 0, 'TRB': 0, 'AST': 0, 'STL': 0, 'BLK': 0, 'TOV': 0,
        'Poss': 0, 'oFGA': 0, 'oFGM': 0, 'o3PA': 0, 'o3PM': 0, 'oFTA': 0, 'oFTM': 0,
        'oORB': 0, 'oTRB': 0, 'oTOV': 0, 'oPoss': 0
    })

    # Initialize/Reset cumulative_stats DataFrame
    cumulative_stats = pd.DataFrame(columns=columns)

    # Loop through each row of data to aggregate stats by team
    for i in tqdm(range(len(data)), desc=year, unit="game"):
        team_name = data.iloc[i]['Team']
        
        # Convert each item in the current row to a float if it looks like a number
        data.iloc[i] = data.iloc[i].apply(lambda x: float(x) if isinstance(x, str) and x.replace('.', '', 1).isdigit() else x)

        # Count the number of successfully converted floats
        float_count = sum(isinstance(value, float) for value in data.iloc[i])

        # Count the number of empty or NaN values in the row
        empty_count = data.iloc[i].isnull().sum()

        # Skip the row if it has no convertible floats or if it has more than 1 empty value
        if float_count == 0 or empty_count > 2:
            continue

        # Update team data in the dictionary
        team_stats[team_name]['Games'] += 1
        team_stats[team_name]['Points'] += data.iloc[i]['Team_Score']
        team_stats[team_name]['FGA'] += data.iloc[i]['FGA']
        team_stats[team_name]['FGM'] += data.iloc[i]['FGM']
        team_stats[team_name]['3PA'] += data.iloc[i]['3PA']
        team_stats[team_name]['3PM'] += data.iloc[i]['3PM']
        team_stats[team_name]['FTA'] += data.iloc[i]['FTA']
        team_stats[team_name]['FTM'] += data.iloc[i]['FTM']
        team_stats[team_name]['ORB'] += data.iloc[i]['ORB']
        team_stats[team_name]['TRB'] += data.iloc[i]['TRB']
        team_stats[team_name]['AST'] += data.iloc[i]['AST']
        team_stats[team_name]['STL'] += data.iloc[i]['STL']
        team_stats[team_name]['BLK'] += data.iloc[i]['BLK']
        team_stats[team_name]['TOV'] += data.iloc[i]['TOV']
        team_stats[team_name]['Poss'] += data.iloc[i]['FGA'] - data.iloc[i]['ORB'] + data.iloc[i]['TOV'] + 0.475 * data.iloc[i]['FTA']
        team_stats[team_name]['oPoints'] += data.iloc[i]['Opp_Score']
        team_stats[team_name]['oFGA'] += data.iloc[i]['oFGA']
        team_stats[team_name]['oFGM'] += data.iloc[i]['oFGM']
        team_stats[team_name]['o3PA'] += data.iloc[i]['o3PA']
        team_stats[team_name]['o3PM'] += data.iloc[i]['o3PM']
        team_stats[team_name]['oFTA'] += data.iloc[i]['oFTA']
        team_stats[team_name]['oFTM'] += data.iloc[i]['oFTM']
        team_stats[team_name]['oORB'] += data.iloc[i]['oORB']
        team_stats[team_name]['oTRB'] += data.iloc[i]['oTRB']
        team_stats[team_name]['oTOV'] += data.iloc[i]['oTOV']
        team_stats[team_name]['oPoss'] += data.iloc[i]['oFGA'] - data.iloc[i]['oORB'] + data.iloc[i]['oTOV'] + 0.475 * data.iloc[i]['oFTA']

        # Convert only the current team's stats to a DataFrame
        current_team_stats = pd.DataFrame([{'Name' : team_name,
                                            'Site' : data.iloc[i]['Site'],
                                            'Date' : data.iloc[i]['Date'],
                                            'Opponent' : data.iloc[i]['Opponent'],
                                            'Outcome' : int(data.iloc[i]['Team_Score'] > data.iloc[i]['Opp_Score']),
                                            'Game' : team_stats[team_name]['Games'],
                                            'ORtg' : 100 * (team_stats[team_name]['Points'] / team_stats[team_name]['Poss']),
                                            'DRtg' : 100 * (team_stats[team_name]['oPoints'] / team_stats[team_name]['oPoss']),
                                            '3PAr' : team_stats[team_name]['3PA'] / team_stats[team_name]['FGA'],
                                            'TS%' : team_stats[team_name]['Points'] / (2 * (team_stats[team_name]['FGA'] + 0.475 * team_stats[team_name]['FTA'])),
                                            'TRB%' : team_stats[team_name]['TRB'] / (team_stats[team_name]['TRB'] + team_stats[team_name]['oTRB']),
                                            'AST%' : team_stats[team_name]['AST'] / team_stats[team_name]['FGM'],
                                            'STL%' : team_stats[team_name]['STL'] / team_stats[team_name]['oPoss'],
                                            'BLK%' : team_stats[team_name]['BLK'] / (team_stats[team_name]['oFGA'] - team_stats[team_name]['o3PA']),
                                            'eFG%' : (team_stats[team_name]['FGM'] + 0.5 * team_stats[team_name]['3PM']) / team_stats[team_name]['FGA'],
                                            'TOV%' : team_stats[team_name]['TOV'] / team_stats[team_name]['Poss'],
                                            'ORB%' : team_stats[team_name]['ORB'] / (team_stats[team_name]['ORB'] + team_stats[team_name]['oTRB'] - team_stats[team_name]['oORB']),
                                            'FTr' : team_stats[team_name]['FTA'] / team_stats[team_name]['FGA'],
                                            'oeFG%' : (team_stats[team_name]['oFGM'] + 0.5 * team_stats[team_name]['o3PM']) / team_stats[team_name]['oFGA'],
                                            'oTOV%' : team_stats[team_name]['TOV'] / team_stats[team_name]['oPoss'],
                                            'oDRB%' : (team_stats[team_name]['oTRB'] - team_stats[team_name]['oORB']) / (team_stats[team_name]['oTRB'] - team_stats[team_name]['oORB'] + team_stats[team_name]['ORB']),
                                            'oFTr' : team_stats[team_name]['oFTA'] / team_stats[team_name]['oFGA'],
                                            'Pace' : (team_stats[team_name]['Poss'] + team_stats[team_name]['oPoss']),
                                            'o3PAr' : team_stats[team_name]['o3PA'] / team_stats[team_name]['oFGA']}])

        cumulative_stats = pd.concat([cumulative_stats, current_team_stats], ignore_index=True)

    # Sort by team name and then by games played within each team
    cumulative_stats.sort_values(by=['Name', 'Game'], inplace=True)

    cumulative_stats.to_csv('AdvWeekData' + year + '.csv', index=False, header=True)


2023:   0%|          | 0/12370 [00:00<?, ?game/s]C:\Users\dschr\AppData\Local\Temp\ipykernel_22732\652547914.py:96: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cumulative_stats = pd.concat([cumulative_stats, current_team_stats], ignore_index=True)
2023: 100%|██████████| 12370/12370 [01:50<00:00, 112.05game/s]


In [30]:
# Creating input data for model

# Define columns for input data
columns = ['Team 1', 'Team 2', 'Date', 'Site', 'Outcome', '1-ORtg', '1-DRtg', '1-3PAr', '1-TS%', '1-TRB%',
           '1-AST%', '1-STL%', '1-BLK%', '1-eFG%', '1-TOV%', '1-ORB%', '1-FTr', '1-oeFG%', '1-oTOV%', '1-oDRB%',
           '1-oFTr', '1-Pace', '1-o3PAr', '2-ORtg', '2-DRtg', '2-3PAr', '2-TS%', '2-TRB%', '2-AST%', '2-STL%',
           '2-BLK%', '2-eFG%', '2-TOV%', '2-ORB%', '2-FTr', '2-oeFG%', '2-oTOV%', '2-oDRB%', '2-oFTr', '2-Pace', '2-o3PAr']

# Initializing dataframe
input_data = pd.DataFrame(columns=columns)

for year in range(2005,2024):
    advanced_stats = pd.read_csv('AdvWeek_CSVs/AdvWeekData' + str(year) + '.csv')

    for j in tqdm(range(len(advanced_stats['Name'])), desc=str(year), unit="game"):
        team_data = advanced_stats.iloc[j]

        name = team_data['Name']
        opp = team_data['Opponent']
        date = team_data['Date']

        # Checking if this game is already in the input_data from the previous team
        row = input_data[(input_data['Team 1'] == opp) & 
                        (input_data['Team 2'] == name) & 
                        (input_data['Date'] == date)]

        # Moving to next iteration if game is already in input_data
        if not row.empty:
            continue

        # Finding opponent's data
        opp_data = advanced_stats[(advanced_stats['Name'] == opp) & (advanced_stats['Date'] == date)]

        # Continue if opponent's data does not exist (Usually a non FBS School)
        if opp_data.empty:
            continue
        else:
            opp_data = opp_data.iloc[0]

        # Getting previous game data
        prev_team_data = advanced_stats[(advanced_stats['Name'] == name) & (advanced_stats['Game'] == int(team_data['Game'])-1)]
        prev_opp_data = advanced_stats[(advanced_stats['Name'] == opp) & (advanced_stats['Game'] == int(opp_data['Game'])-1)]

        if prev_team_data.empty or prev_opp_data.empty:
            continue

        prev_opp_data = prev_opp_data.iloc[0]
        prev_team_data = prev_team_data.iloc[0]

        current_stats = pd.DataFrame([{'Team 1' : team_data['Name'],
                                        'Team 2' : team_data['Opponent'],
                                        'Date' : team_data['Date'],
                                        'Site' : team_data['Site'],
                                        'Outcome' : team_data['Outcome'],
                                        '1-ORtg' : prev_team_data['ORtg'],
                                        '1-DRtg' : prev_team_data['DRtg'],
                                        '1-3PAr' : prev_team_data['3PAr'],
                                        '1-TS%' : prev_team_data['TS%'],
                                        '1-TRB%' : prev_team_data['TRB%'],
                                        '1-AST%' : prev_team_data['AST%'],
                                        '1-STL%' : prev_team_data['STL%'],
                                        '1-BLK%' : prev_team_data['BLK%'],
                                        '1-eFG%' : prev_team_data['eFG%'],
                                        '1-TOV%' : prev_team_data['TOV%'],
                                        '1-ORB%' : prev_team_data['ORB%'],
                                        '1-FTr' : prev_team_data['FTr'],
                                        '1-oeFG%' : prev_team_data['oeFG%'],
                                        '1-oTOV%' : prev_team_data['oTOV%'],
                                        '1-oDRB%' : prev_team_data['oDRB%'],
                                        '1-oFTr' : prev_team_data['oFTr'],
                                        '1-Pace' : prev_team_data['Pace'],
                                        '1-o3PAr' : prev_team_data['o3PAr'],
                                        '2-ORtg' : prev_opp_data['ORtg'],
                                        '2-DRtg' : prev_opp_data['DRtg'],
                                        '2-3PAr' : prev_opp_data['3PAr'],
                                        '2-TS%' : prev_opp_data['TS%'],
                                        '2-TRB%' : prev_opp_data['TRB%'],
                                        '2-AST%' : prev_opp_data['AST%'],
                                        '2-STL%' : prev_opp_data['STL%'],
                                        '2-BLK%' : prev_opp_data['BLK%'],
                                        '2-eFG%' : prev_opp_data['eFG%'],
                                        '2-TOV%' : prev_opp_data['TOV%'],
                                        '2-ORB%' : prev_opp_data['ORB%'],
                                        '2-FTr' : prev_opp_data['FTr'],
                                        '2-oeFG%' : prev_opp_data['oeFG%'],
                                        '2-oTOV%' : prev_opp_data['oTOV%'],
                                        '2-oDRB%' : prev_opp_data['oDRB%'],
                                        '2-oFTr' : prev_opp_data['oFTr'],
                                        '2-Pace' : prev_opp_data['Pace'],
                                        '2-o3PAr' : prev_opp_data['o3PAr']}])

        input_data = pd.concat([input_data, current_stats], ignore_index=True)

# Sort games by Date
input_data.sort_values(by=['Date'], inplace=True)

# Export Data to a csv
input_data.to_csv('InputData.csv', index=False, header=True)


2005:   0%|          | 0/8836 [00:00<?, ?game/s]C:\Users\dschr\AppData\Local\Temp\ipykernel_22732\1122947695.py:92: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  input_data = pd.concat([input_data, current_stats], ignore_index=True)
2023: 100%|██████████| 12370/12370 [04:20<00:00, 47.53game/s]
